# 1.01 Python 与 NumPy 速成

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/01_prerequisites/1.01_python_numpy.ipynb)


## 为什么要学

VLA 项目里大量的代码不是"模型"，而是**数据处理**：读取轨迹、切动作块、算归一化统计量、筛选成功的回合。这些代码几乎全是 Python + NumPy。NumPy 的思维方式（向量化、广播）和 PyTorch 完全一样，学会它就等于学会了一半 PyTorch。

本课只讲写 VLA 必须的部分，不求全。

---

## 1. Python：写模型必须会的四样东西

### 1.1 列表推导式与字典

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
# 列表推导式：一行生成列表
squares = [i * i for i in range(5)]
evens = [i for i in range(10) if i % 2 == 0]
print(squares, evens)

# 字典：VLA 里的"观测"几乎都是字典
obs = {
    "image": "一张图片",
    "state": [0.1, 0.2, 0.3],
    "instruction": "去红色方块",
}
for key, value in obs.items():
    print(f"{key:12s} -> {value}")

### 1.2 函数：默认参数、关键字参数、`*args / **kwargs`

In [ ]:
def make_config(lr=1e-3, batch_size=32, **kwargs):
    """**kwargs 收集所有"多余的"关键字参数，常用于把配置一路传下去。"""
    cfg = {"lr": lr, "batch_size": batch_size}
    cfg.update(kwargs)
    return cfg

print(make_config())
print(make_config(lr=3e-4, horizon=50, use_wrist_cam=True))

### 1.3 类：`nn.Module` 的基础

PyTorch 中**每个模型都是一个类**。你必须看懂 `__init__`（创建时运行一次，用来定义"零件"）和普通方法（每次调用时运行，用来定义"计算"）。

In [ ]:
class RunningMean:
    """在线计算均值：每来一个新数，更新一次，不需要保存所有历史数据。

    这个技巧在强化学习里很常用（例如统计回报的均值做归一化）。
    """

    def __init__(self):
        self.n = 0        # 已经见过多少个数
        self.mean = 0.0   # 当前均值

    def update(self, x):
        self.n += 1
        # 新均值 = 旧均值 + (新数 - 旧均值) / n   （推导：把 n 个数的均值拆成前 n-1 个和第 n 个）
        self.mean += (x - self.mean) / self.n
        return self.mean

rm = RunningMean()
for x in [2, 4, 6, 8]:
    print("来了", x, "当前均值", rm.update(x))

### 1.4 `dataclass`：配置的标准写法

openpi 的所有配置（`TrainConfig`、`Pi0Config`）都是 `dataclass`。它自动帮你写 `__init__`。

In [ ]:
from dataclasses import dataclass, field

@dataclass
class PolicyConfig:
    action_dim: int = 7          # 动作维度
    action_horizon: int = 50     # 动作块长度
    hidden: int = 256
    camera_names: list = field(default_factory=lambda: ["base", "wrist"])  # 可变默认值要用 field

cfg = PolicyConfig(action_dim=14)   # 双臂 ALOHA：14 维
print(cfg)
print(cfg.action_horizon)

---

## 2. NumPy：数组思维

### 2.1 数组的三个属性：`shape`、`dtype`、`ndim`

In [ ]:
import numpy as np

a = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.float32)
print("shape:", a.shape)   # (2, 3)：2 行 3 列
print("dtype:", a.dtype)   # float32
print("ndim :", a.ndim)    # 2 维

# 常用创建方式
print(np.zeros((2, 3)))
print(np.arange(0, 1, 0.25))
print(np.linspace(0, 1, 5))
rng = np.random.default_rng(0)       # 推荐的随机数写法：用一个生成器对象，方便固定种子
print(rng.standard_normal((2, 2)))

### 2.2 索引与切片

In [ ]:
traj = np.arange(20).reshape(10, 2)    # 一条 10 步的轨迹，每步 2 维
print(traj[0])        # 第 0 步           -> shape (2,)
print(traj[-1])       # 最后一步
print(traj[2:5])      # 第 2、3、4 步      -> shape (3, 2)
print(traj[:, 0])     # 所有步的第 0 维    -> shape (10,)
print(traj[::3])      # 每隔 3 步取一个（降采样）

# 布尔索引：筛选满足条件的元素（数据清洗最常用）
x = np.array([3, -1, 4, -1, 5])
print(x[x > 0])

# 高级索引：用整数数组一次取多个位置
idx = np.array([0, 3, 9])
print(traj[idx])

### 2.3 广播（broadcasting）：最重要的概念

两个形状不同的数组做运算时，NumPy 会自动"扩展"它们。规则只有一条：

> **从最后一维开始往前对齐，每一维要么相等，要么其中一个是 1（或者不存在）。是 1 的那一维会被"复制"成另一个的大小。**

| A 的形状 | B 的形状 | 结果 | 说明 |
|---|---|---|---|
| `(5, 3)` | `(3,)` | `(5, 3)` | B 被复制 5 行 |
| `(5, 1)` | `(1, 4)` | `(5, 4)` | 两个都被扩展 |
| `(5, 3)` | `(5,)` | ❌ 报错 | 最后一维 3 vs 5 不匹配 |
| `(5, 3)` | `(5, 1)` | `(5, 3)` | 用 `[:, None]` 增加维度 |

In [ ]:
X = np.random.randn(5, 3)
mean = X.mean(axis=0)            # (3,)：每一列的均值
X_centered = X - mean            # (5, 3) - (3,) -> (5, 3)
print(X_centered.mean(axis=0).round(6))   # 每列均值变成 0

row_sum = X.sum(axis=1)          # (5,)
# X / row_sum 会报错；需要把 row_sum 变成 (5, 1)
X_rownorm = X / row_sum[:, None]
print(X_rownorm.shape)

### 2.4 `axis` 的含义

`axis=k` 表示**沿着第 k 维做运算，并把这一维"压扁"**。形状 `(B, T, D)` 的数组：

- `.mean(axis=0)` → `(T, D)`：对 batch 求平均；
- `.mean(axis=1)` → `(B, D)`：对时间求平均；
- `.mean(axis=-1)` → `(B, T)`：对特征求平均。

In [ ]:
data = np.random.randn(8, 50, 7)   # 8 条轨迹，每条 50 步，7 维动作
print(data.mean(axis=0).shape, data.mean(axis=1).shape, data.mean(axis=-1).shape)
# keepdims=True 保留被压扁的那一维（长度变成 1），方便后续广播
print(data.mean(axis=-1, keepdims=True).shape)

---

## 练习

### 练习 1：成对距离矩阵（广播）

给定 `N` 个点 `P (N, 2)` 和 `M` 个点 `Q (M, 2)`，返回距离矩阵 `D (N, M)`，其中 `D[i, j] = ||P[i] - Q[j]||`。**不要用 for 循环**。

> 这个操作在 RL 里用来算"夹爪离每个物体多远"，在注意力机制里有同样的"两两比较"结构。

In [ ]:
def pairwise_dist(P, Q):
    """P: (N, 2), Q: (M, 2) -> (N, M)"""
    # TODO: 提示：P[:, None, :] 的形状是 (N, 1, 2)，Q[None, :, :] 是 (1, M, 2)
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def pairwise_dist(P, Q):
    """P: (N, 2), Q: (M, 2) -> (N, M)"""
    diff = P[:, None, :] - Q[None, :, :]     # (N, 1, 2) - (1, M, 2) -> (N, M, 2)
    return np.sqrt((diff ** 2).sum(axis=-1))  # (N, M)

In [ ]:
# ✅ 自测
P = np.random.randn(4, 2); Q = np.random.randn(6, 2)
D = pairwise_dist(P, Q)
assert D.shape == (4, 6)
assert np.isclose(D[2, 5], np.linalg.norm(P[2] - Q[5]))
print("通过 ✅")

### 练习 2：动作归一化统计量

给定一个数据集里的所有动作 `actions (N, D)`，计算每一维的**均值、标准差、1% 分位数、99% 分位数**，返回一个字典。

> openpi 的 `compute_norm_stats.py` 就是在做这件事：它统计 `mean/std/q01/q99`，π0.5 用分位数把动作缩放到 [-1, 1]（比均值方差更不怕离群值）。

In [ ]:
def compute_norm_stats(actions):
    """actions: (N, D) -> dict(mean=(D,), std=(D,), q01=(D,), q99=(D,))"""
    # TODO: 用 np.mean / np.std / np.quantile，注意 axis
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def compute_norm_stats(actions):
    """actions: (N, D) -> dict(mean=(D,), std=(D,), q01=(D,), q99=(D,))"""
    return {
        "mean": actions.mean(axis=0),
        "std": actions.std(axis=0),
        "q01": np.quantile(actions, 0.01, axis=0),
        "q99": np.quantile(actions, 0.99, axis=0),
    }

In [ ]:
# ✅ 自测
acts = np.random.default_rng(0).normal(loc=[0, 5], scale=[1, 2], size=(10000, 2))
st = compute_norm_stats(acts)
assert st["mean"].shape == (2,)
assert abs(st["mean"][1] - 5) < 0.1 and abs(st["std"][1] - 2) < 0.1
assert st["q01"][0] < -2 and st["q99"][0] > 2
print("通过 ✅", {k: v.round(2) for k, v in st.items()})

### 练习 3：分位数归一化（π0.5 的做法）

利用上面的 `q01, q99`，把动作线性映射到 `[-1, 1]`：`q01 → -1`，`q99 → 1`。再写出**反归一化**（把模型输出变回真实动作，部署时必须做这一步）。

$$
\hat a = 2\cdot\frac{a - q_{01}}{q_{99} - q_{01}} - 1, \qquad a = \frac{(\hat a + 1)(q_{99} - q_{01})}{2} + q_{01}
$$

In [ ]:
def quantile_normalize(a, q01, q99):
    # TODO
    raise NotImplementedError

def quantile_unnormalize(a_hat, q01, q99):
    # TODO
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def quantile_normalize(a, q01, q99):
    return 2.0 * (a - q01) / (q99 - q01 + 1e-6) - 1.0

def quantile_unnormalize(a_hat, q01, q99):
    return (a_hat + 1.0) / 2.0 * (q99 - q01 + 1e-6) + q01

In [ ]:
# ✅ 自测
a_hat = quantile_normalize(acts, st["q01"], st["q99"])
print("归一化后 1%/99% 分位数:", np.quantile(a_hat, [0.01, 0.99], axis=0).round(3))
assert np.allclose(quantile_unnormalize(a_hat, st["q01"], st["q99"]), acts, atol=1e-6)
print("通过 ✅ 归一化 -> 反归一化 能还原原始动作")

---

## 小结

- Python：会用字典、`**kwargs`、类和 `dataclass`，就足以读懂 openpi 的配置和模型代码。
- NumPy 的核心是**广播**与 **axis**：想清楚每一维的含义，就能把循环写成一行。
- 归一化/反归一化是 VLA 部署时最容易出 bug 的地方之一：训练用了什么统计量，部署时必须用**同一份**。

**思考题**：如果某个关节在训练数据里几乎没动过（`q99 ≈ q01`），分位数归一化会出什么问题？代码里的 `1e-6` 够吗？（提示：openpi 对这种情况会怎么处理？可以去 `openpi/src/openpi/transforms.py` 里搜 `Normalize` 看看。）